In [ ]:
import tensorflow as tf  
# For keras like progress bars in the training loop :


class BaseModel:
    
    def __init__(self, ckpt_name='model.pt'):
        self.ckpt_name = ckpt_name
        self.start_epoch = 0
        self.best_loss = 10000
    
    def on_val_end(self):
        val_loss = self.val_dict[self.monitor_metric]
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.save()
            
    def save(self):
        raise NotImplementedError
            
            
    def on_train_start(self):
        pass
    
    
    def on_epoch_end(self):
        pass
    
    
    def on_fit_end(self):
        pass
    
    
    def on_epoch_start(self):
        pass
                        
                        
    def fit(self, dl, valid_dl=None, monitor_metric='val_loss', n_epochs=1):
        self.dl = dl
        self.valid_dl = valid_dl
        self.n_epochs = n_epochs
        self.monitor_metric = monitor_metric
        
        self.on_train_start()
        for epoch in range(n_epochs):
            
            self.on_epoch_start()
            self.epoch = epoch
            self.n_batches = len(dl)
            print(f'Epoch {epoch+1}/{n_epochs}')
            pbar = tf.keras.utils.Progbar(target=self.n_batches)
            
            for idx, batch in enumerate(dl):
                
                self.batch_idx = idx
                loss_dict = self.train_step(epoch, idx, batch) 
                pbar.update(idx, values=list(loss_dict.items()))
                
            if valid_dl:
                self.validate()
                pbar.update(self.n_batches, values=list(self.val_dict.items()))
                self.on_val_end()
            else:
                pbar.update(self.n_batches, values=None)
            
            self.on_epoch_end()
            
        self.on_fit_end()

In [ ]:
import torch
import random
    
class ProtoData:
    def __init__(self, ds, nshot, nquery, nway, labels, num_batches):
        self.nway = nway
        self.nshot = nshot
        self.nquery = nquery
        self.labels = labels
        self.num_batches = num_batches
        self.ds = ds
        self.d1 = ds[0][0].shape[0]
        self.d2 = ds[0][0].shape[1]
        self.d3 = ds[0][0].shape[2]
        label_idx = {}
        for i in range(len(ds)):
            l = ds[i][1]
            if l not in label_idx:
                label_idx[l] = []
            label_idx[l].append(i)
        self.label_idx = label_idx
        
    def __len__(self):
        return self.num_batches
    
    def __getitem__(self, idx):
        """
        batch = [
            class1_1,  class1_2, ... , class2_1, class2_2, ... # supports
            class1_query1, class1_query2,...,class2_query1, class1_query2,...]
        """
        select_labels = random.sample(self.labels, self.nway)
        if idx >= self.num_batches:
            raise IndexError
        bs = self.nway * self.nshot + self.nquery * self.nway
        batch = torch.zeros(bs,self.d1, self.d2, self.d3)
        for class_idx,c in enumerate(select_labels):
            shuffled_idx = random.sample(self.label_idx[c], len(self.label_idx[c]))
            selection = shuffled_idx[:self.nshot + self.nquery]
            for selection_idx,i in enumerate(selection):
                if selection_idx < self.nshot:
                    batch[class_idx*self.nshot+selection_idx,:,:,:] = self.ds[i][0]
                else:
                    batch[self.nshot*self.nway + class_idx*self.nquery + (selection_idx - self.nshot),:,:,:] = self.ds[i][0]
        return batch

In [ ]:
import torchvision

ds = torchvision.datasets.Omniglot(
    root='.', download=True, transform=torchvision.transforms.ToTensor()
)

from matplotlib.pyplot import imshow
import matplotlib.pyplot as plt

def tensor_to_img(x):
    data = x.squeeze().numpy() 
    plt.figure()
    plt.imshow(data, cmap='gray', vmin=0, vmax=1)

tensor_to_img(ds[900][0])  # plot single image

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class ProtoNet(BaseModel):
    def __init__(self, enc, nshot, nway, nquery, lr=0.001):
        super().__init__()
        self.enc = enc
        self.nshot = nshot
        self.nway = nway
        self.nquery = nquery
        self.opt = torch.optim.Adam(self.enc.parameters(), lr=lr)
        self.loss_fn = torch.nn.NLLLoss()

    def on_train_start(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.loss_fn.to(self.device)
        self.enc = nn.DataParallel(self.enc).to(self.device)
        self.epoch = -1
        
    def compute_prototypes(self, support, k, n):
        class_prototypes = support.reshape(k, n, -1).mean(dim=1)
        return class_prototypes
    
    def pairwise_distances(self, x, y):
        """ Calculate l2 distance between each element of x and y.
        Cosine similarity can also be used
        """
        n_x = x.shape[0]
        n_y = y.shape[0]
        
        distances = (
            x.unsqueeze(1).expand(n_x, n_y, -1) -
            y.unsqueeze(0).expand(n_x, n_y, -1)
        ).pow(2).sum(dim=2)
        return distances

    def train_step(self, epoch, idx, x):
        self.enc.train()
        x = x.to(self.device)
        embeddings = self.enc(x)

        support = embeddings[:self.nshot*self.nway]
        queries = embeddings[self.nshot*self.nway:]
        prototypes = self.compute_prototypes(support, self.nway, self.nshot)
        
        distances = self.pairwise_distances(queries, prototypes)  # (num_queries, k_way)

        # Calculate logits with softmax
        log_p_y = (-distances).log_softmax(dim=1)
        
        # labels = [class1 * nquery, class2 * nquery, ...]
        y = []
        for c in range(self.nway):
            y.extend([c]*self.nquery)
        y = torch.tensor(y).to(self.device)
        loss = self.loss_fn(log_p_y, y)
        
        self.opt.zero_grad()
        loss.backward()
        self.opt.step()
        return {"loss": loss.item()}

    def save(self):
        torch.save(
            {
                "epoch": self.epoch+1+self.start_epoch,
                "enc": self.enc.state_dict(),
                "opt": self.opt.state_dict()
            },
            self.ckpt_name,
        )
        
    def load(self, ckpt_path):
        ckpt = torch.load(ckpt_path)
        self.opt.load_state_dict(ckpt['opt'])
        self.enc.load_state_dict(ckpt['enc'])
        self.start_epoch = ckpt['epoch']

    def on_epoch_end(self):
        if not hasattr(self, 'test_dl'):
            print('No test dataloader')
            return
        
        nshot = self.test_nshot
        nquery = self.test_nquery
        nway = self.test_nway
        self.enc.eval()
        acc = []
        bs = []
        for x in self.test_dl:
            x = x.to(self.device)
            with torch.no_grad():
                embeddings = self.enc(x)

            support = embeddings[:nshot*nway]
            queries = embeddings[nshot*nway:]
            prototypes = self.compute_prototypes(support, nway, nshot)

            distances = self.pairwise_distances(queries, prototypes, 'l2')

            log_p_y = (-distances).log_softmax(dim=1)

            y_pred = (-distances).softmax(dim=1)
            preds = torch.argmax(y_pred, dim=1)
            
            y = []
            for c in range(nway):
                y.extend([c]*nquery)
            y = torch.tensor(y).to(self.device)
            
            batch_acc = (preds==y).cpu().float().mean().item()
            acc.append(batch_acc)
            bs.append(x.shape[0])

        numerator = sum([size * _ for size,_ in zip(bs,acc)])
        denominator = sum(bs)
        acc = numerator / denominator
        print(f'epoch {self.epoch+1}: few shot accuracy {acc:.4f}')

In [ ]:
def conv_block(in_channels, out_channels):
    bn = nn.BatchNorm2d(out_channels)
    nn.init.uniform_(bn.weight)
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        bn,
        nn.ReLU(),
        nn.MaxPool2d(2)
    )


class Convnet(nn.Module):

    def __init__(self, x_dim=1, hid_dim=64, z_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            conv_block(x_dim, hid_dim),
            conv_block(hid_dim, hid_dim),
            conv_block(hid_dim, hid_dim),
            conv_block(hid_dim, hid_dim),
            conv_block(hid_dim, hid_dim),
            conv_block(hid_dim, z_dim),
        )
        self.fc1 = nn.Linear(64,64)
        self.fc2 = nn.Linear(64,32)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        x = self.fc2(self.relu(self.fc1(x)))
        return x
        

enc = Convnet()
nshot = 1
nway = 60
test_nway = 5
nquery = 5
model = ProtoNet(enc, nshot=nshot, nway=nway, nquery=nquery, lr=0.001)
proto_loader = ProtoData(
    ds, 
    nshot=nshot, 
    nquery=nquery, 
    nway=nway, 
    labels=[i for i in range(900)], # labels 0-899 for training
    num_batches=100
)
model.test_dl = ProtoData(
    ds, 
    nshot=nshot, 
    nquery=nquery, 
    nway=test_nway, 
    labels=[(901 + i) for i in range(50)], # labels 901-950 for testing
    num_batches=100
)
model.test_nshot = nshot
model.test_nway = test_nway
model.test_nquery = nquery

In [ ]:
model.fit(proto_loader, n_epochs=10)